In [4]:
import pandas as pd

# Carga y Normalización Espejo
df_mayor = pd.read_csv("../data/raw/libro_mayor.csv")
df_banco = pd.read_csv("../data/raw/extracto_bancario.csv")

df_mayor['fecha_contable'] = pd.to_datetime(df_mayor['fecha_contable'])
df_banco['fecha_banco'] = pd.to_datetime(df_banco['fecha_banco'])

# Espejo bancario: multiplicamos por -1 para comparar 1:1 con el Mayor
df_banco['monto_normalizado'] = df_banco['monto'] * -1

In [5]:
# CLASIFICACIÓN DE AUDITORÍA INTEGRAL
todas_partidas = pd.merge(
    df_mayor,
    df_banco,
    on='ref_comprobante',
    how='outer',
    suffixes=('_mayor', '_banco')
)

todas_partidas['dias_clearing'] = (todas_partidas['fecha_banco'] - todas_partidas['fecha_contable']).dt.days

def clasificar_partida(row):
    # Caso 1 a 4: Comprobante en ambas fuentes
    if pd.notna(row['fecha_contable']) and pd.notna(row['fecha_banco']):
        if row['monto_mayor'] == row['monto_normalizado']:
            if 0 <= row['dias_clearing'] <= 3:
                return 'CONCILIADO_DENTRO_DE_CLEARING'
            elif row['dias_clearing'] < 0:
                return 'ALERTA_BANCO_ANTERIOR_A_MAYOR'
            else:
                return 'CONCILIADO_FUERA_DE_TERMINO'
        else:
            return 'DISCREPANCIA_DE_MONTO'
    # Caso 5: Pendiente en Banco (Solo en Mayor)
    elif pd.notna(row['fecha_contable']) and pd.isna(row['fecha_banco']):
        return 'PENDIENTE_ACREDITACION_BANCO'
    # Caso 6: Omitido en Mayor (Solo en Banco)
    elif pd.isna(row['fecha_contable']) and pd.notna(row['fecha_banco']):
        return 'OMISION_REGISTRO_MAYOR'

    return 'REVISION_MANUAL'

todas_partidas['estado_audit'] = todas_partidas.apply(clasificar_partida, axis=1)

print("📊 Distribución Exacta por Estado de Auditoría:")
print(todas_partidas['estado_audit'].value_counts())

📊 Distribución Exacta por Estado de Auditoría:
estado_audit
CONCILIADO_DENTRO_DE_CLEARING    799
CONCILIADO_FUERA_DE_TERMINO      180
ALERTA_BANCO_ANTERIOR_A_MAYOR     10
PENDIENTE_ACREDITACION_BANCO      10
OMISION_REGISTRO_MAYOR             3
DISCREPANCIA_DE_MONTO              1
Name: count, dtype: int64


In [6]:
# Exportación de Evidencias
import os

ruta_processed = "../data/processed"
os.makedirs(ruta_processed, exist_ok=True)

todas_partidas.to_csv(f"{ruta_processed}/conciliados_final.csv", index=False)

# Partidas para revisión contable
pendientes_mayor = todas_partidas[todas_partidas['estado_audit'] == 'PENDIENTE_ACREDITACION_BANCO']
pendientes_banco = todas_partidas[todas_partidas['estado_audit'] == 'OMISION_REGISTRO_MAYOR']
discrepancias = todas_partidas[todas_partidas['estado_audit'] == 'DISCREPANCIA_DE_MONTO']

with pd.ExcelWriter(f"{ruta_processed}/Partidas_Pendientes_Marzo.xlsx", engine='openpyxl') as writer:
    pendientes_mayor.to_excel(writer, sheet_name='Revisar_en_Mayor', index=False)
    pendientes_banco.to_excel(writer, sheet_name='Revisar_en_Banco', index=False)
    discrepancias.to_excel(writer, sheet_name='Discrepancias_Monto', index=False)

print("✅ Evidencia exportada a data/processed/ exitosamente.")

✅ Evidencia exportada a data/processed/ exitosamente.
